In [ ]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
from __future__ import unicode_literals

import warnings
warnings.filterwarnings('ignore')
import numpy as np

from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
from torch.optim.lr_scheduler import LambdaLR
import torch
from torch import nn
import torch.nn.functional as F
from torch import optim
from torch.autograd import Variable
from torch.cuda.amp import GradScaler, autocast

import tqdm
import pickle
import argparse
import random
import math
import os
import bisect
import collections
from os.path import exists, getsize, join


import dill


from sklearn.utils import shuffle

use_cuda = torch.cuda.is_available()

if use_cuda:
    n_gpu = torch.cuda.device_count()
    device = torch.device("cuda")
else:
    n_gpu = 0
    device = torch.device("cpu")

kwargs = {'num_workers': 2, 'pin_memory': True} if use_cuda else {}

print("Device:", device)
print("GPUs available:", n_gpu)

for i in range(n_gpu):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


# Cell 2: Configuration
# Cấu hình đường dẫn
INPUT_TRAFFIC_PATH = '/kaggle/input/datasets/hphglinh/mac-new/50labels500_mac_new'  # Folder chứa tất cả file traffic
OUTPUT_PATH = '/kaggle/working/output/'  # Đường dẫn output

# Tham số
MAX_LABELS = 50  # Số lượng labels tối đa muốn extract

print(f"Input path: {INPUT_TRAFFIC_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Max labels to extract: {MAX_LABELS}")

In [ ]:
batch_size = 128
fp16_precision = True
temperature = 0.5
n_views = 2
num_epoches = 100

In [ ]:
def extract_label_from_filename(filename):
    name = filename.replace('.txt', '')
    parts = name.split('_')
    
    # Find the index of the last numeric part and use everything before it as the label
    label_parts = []
    for i, part in enumerate(parts):
        if part.isdigit():
            label_parts = parts[:i]
            break
    else:
        label_parts = parts
    
    label = '_'.join(label_parts)
    if label.startswith('traffic_'):
        label = label[8:]
    
    return label


def scan_traffic_files(input_path):
    """Scan all files, extract unique labels, and map them to indices."""
    
    if not os.path.exists(input_path):
        print(f"❌ Path not found: {input_path}")
        return {}, {}
    
    all_files = [f for f in os.listdir(input_path) if f.endswith('.txt')]
    
    label_set = set()
    file_label_map = {}
    
    for filename in all_files:
        label = extract_label_from_filename(filename)
        label_set.add(label)
        file_label_map[filename] = label
    
    sorted_labels = sorted(list(label_set))
    
    # Cap the number of labels at MAX_LABELS
    if len(sorted_labels) > MAX_LABELS:
        sorted_labels = sorted_labels[:MAX_LABELS]
    
    label_mapping = {label: idx for idx, label in enumerate(sorted_labels)}
    
    # Keep only files whose labels are within the mapping
    filtered_file_label_map = {
        filename: label 
        for filename, label in file_label_map.items() 
        if label in label_mapping
    }
    
    return label_mapping, filtered_file_label_map

In [ ]:
# Cell 5: Helper Functions - Get Local IP
def get_local_ip(path):
    ip_list = []
    with open(path, 'r') as file:
        packets = file.readlines()

    for packet_line in packets:
        packet = packet_line.strip()
        if not packet:
            continue
        strs = packet.split(',')
        if len(strs) < 6:
            continue
        timestamp, src_ip, sport, dst_ip, dport, packet_size = strs[:6]
        ip_list.append(src_ip)
        ip_list.append(dst_ip)

    counter = collections.Counter(ip_list)
    if len(counter) == 0:
        return -1
    local_ip = counter.most_common(1)[0][0]
    return local_ip

print("✓ get_local_ip() defined")

In [ ]:
from scipy import stats
import numpy as np
import pandas as pd
import os
from os.path import join, exists, getsize
import datetime


def parse_pcap_direction(path):
    """Parse traffic file and extract packet directions only."""
    local_ip = get_local_ip(path)
    directions = []
    
    if local_ip == -1:
        return directions

    with open(path, 'r') as file:
        packets = file.readlines()

    for packet_line in packets:
        packet = packet_line.strip()
        if not packet:
            continue
        strs = packet.split(',')
        if len(strs) < 6:
            continue
            
        timestamp, src_ip, sport, dst_ip, dport, packet_size = strs[:6]
        
        # -1 = outgoing, +1 = incoming
        direction = -1 if src_ip == local_ip else 1
        directions.append(direction)
    
    return directions


def generate_direction_vector(traffic_file_path, label_id, max_packets, min_packets=100):
    """Generate a direction vector with zero-padding (-1 out, +1 in)."""
    if not exists(traffic_file_path):
        return None, 0
    if getsize(traffic_file_path) == 0:
        return None, 0
        
    try:
        directions = parse_pcap_direction(traffic_file_path)
        packet_count = len(directions)
        
        if packet_count < min_packets:
            return None, 0
        
        if packet_count > max_packets:
            directions = directions[:max_packets]
        
        # Build vector with zero-padding then fill actual directions
        direction_vector = np.zeros(max_packets, dtype=int)
        for i, direction in enumerate(directions):
            direction_vector[i] = direction
        
        sample = [label_id] + direction_vector.tolist()
        return sample, packet_count
        
    except Exception as e:
        return None, 0


def process_direction_data(input_traffic_path, output_path):
    """Process all traffic data into direction vectors and save to CSV."""
    
    os.makedirs(output_path, exist_ok=True)
    
    label_mapping, file_label_map = scan_traffic_files(input_traffic_path)
    
    if len(label_mapping) == 0:
        return
    
    max_packets = 10000

    # Group files by label
    files_by_label = {}
    for filename, label_name in file_label_map.items():
        if label_name not in files_by_label:
            files_by_label[label_name] = []
        files_by_label[label_name].append(filename)
    
    # Remove labels with fewer than MIN_FILES_PER_LABEL files
    MIN_FILES_PER_LABEL = 50
    filtered_labels = {
        label: files for label, files in files_by_label.items()
        if len(files) >= MIN_FILES_PER_LABEL
    }
    
    if len(filtered_labels) == 0:
        return
    
    # Rebuild label mapping from filtered labels
    new_label_mapping = {
        label: i for i, label in enumerate(sorted(filtered_labels.keys()))
    }
    
    label_df = pd.DataFrame(list(new_label_mapping.items()), columns=['label_name', 'label_id'])
    label_df.to_csv(join(output_path, 'label_mapping.csv'), index=False)
    
    all_samples = []
    
    for label_name in sorted(filtered_labels.keys()):
        label_id = new_label_mapping[label_name]
        files = sorted(filtered_labels[label_name])
        
        for filename in files:
            traffic_file_path = join(input_traffic_path, filename)
            result = generate_direction_vector(traffic_file_path, label_id, max_packets)
            
            if result is not None and result[0] is not None:
                sample, _ = result
                all_samples.append(sample)
    
    if all_samples:
        output_file = join(output_path, 'direction_vectors.csv')
        df = pd.DataFrame(all_samples)
        df.to_csv(output_file, header=False, index=False)


# Run extraction
process_direction_data(INPUT_TRAFFIC_PATH, OUTPUT_PATH)

# Verify output
direction_file = join(OUTPUT_PATH, 'direction_vectors.csv')

if os.path.exists(direction_file):
    df = pd.read_csv(direction_file, header=None)
    print(f"Total samples: {len(df):,}")
    print(f"Vector length: {len(df.columns)} (1 label + {len(df.columns)-1} directions)")
    print(f"File size: {os.path.getsize(direction_file) / 1024 / 1024:.2f} MB")
    print(df[0].value_counts().sort_index())

In [ ]:
def load_direction_data(direction_file):
    """Load direction vectors and split into x_train, y_train."""
    if not os.path.exists(direction_file):
        return None, None
    
    df = pd.read_csv(direction_file, header=None)
    
    # Column 0 is the label, the rest are features
    y_train = df.iloc[:, 0].values
    x_train = df.iloc[:, 1:].values
    
    return x_train, y_train


def sample_for_pretrain(x_data, y_data, n_samples_per_class):
    """Sample N samples per class for pretraining."""
    x_sampled = []
    y_sampled = []
    
    for label in np.unique(y_data):
        indices = np.where(y_data == label)[0]
        sampled_indices = np.random.choice(
            indices, size=min(n_samples_per_class, len(indices)), replace=False
        )
        x_sampled.append(x_data[sampled_indices])
        y_sampled.append(y_data[sampled_indices])
    
    return np.concatenate(x_sampled), np.concatenate(y_sampled)


# Load and sample data
direction_file = join(OUTPUT_PATH, 'direction_vectors.csv')
x_train, y_train = load_direction_data(direction_file)
x_train, y_train = sample_for_pretrain(x_train, y_train, n_samples_per_class=250)

In [ ]:
class DFNet(nn.Module):
    def __init__(self, out_dim):
        super(DFNet, self).__init__()
        kernel_size = 8
        channels = [1, 32, 64, 128, 256]
        conv_stride = 1
        pool_stride = 4
        pool_size = 8
        
        self.conv1 = nn.Conv1d(1, 32, kernel_size, stride = conv_stride)
        self.conv1_1 = nn.Conv1d(32, 32, kernel_size, stride = conv_stride)
        
        self.conv2 = nn.Conv1d(32, 64, kernel_size, stride = conv_stride)
        self.conv2_2 = nn.Conv1d(64, 64, kernel_size, stride = conv_stride)
       
        self.conv3 = nn.Conv1d(64, 128, kernel_size, stride = conv_stride)
        self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride = conv_stride)
       
        self.conv4 = nn.Conv1d(128, 256, kernel_size, stride = conv_stride)
        self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride = conv_stride)
       
        
        self.batch_norm1 = nn.BatchNorm1d(32)
        self.batch_norm2 = nn.BatchNorm1d(64)
        self.batch_norm3 = nn.BatchNorm1d(128)
        self.batch_norm4 = nn.BatchNorm1d(256)
        
        self.max_pool_1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_2 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_3 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_4 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        
        self.dropout1 = nn.Dropout(p=0.1)
        self.dropout2 = nn.Dropout(p=0.1)
        self.dropout3 = nn.Dropout(p=0.1)
        self.dropout4 = nn.Dropout(p=0.1)

        
        self.fc = nn.Linear(10240, out_dim)

        
    def weight_init(self):
        for n, m in self.named_modules():
            if isinstance(m, nn.Linear) or isinstance(m, nn.Conv1d):
#                 m.weight.data.xavier_uniform_()
                # print (n)
                torch.nn.init.xavier_uniform(m.weight)
                m.bias.data.zero_()
            
        
    def forward(self, inp):
        x = inp
        # ==== first block ====
        x = F.pad(x, (3,4))
        x = F.elu((self.conv1(x)))
        x = F.pad(x, (3,4))
        x = F.elu(self.batch_norm1(self.conv1_1(x)))
#         x = F.elu(self.conv1_1(x))
        x = F.pad(x, (3, 4))
        x = self.max_pool_1(x)
        x = self.dropout1(x)
        
        # ==== second block ====
        x = F.pad(x, (3,4))
        x = F.relu((self.conv2(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm2(self.conv2_2(x)))
#         x = F.relu(self.conv2_2(x))
        x = F.pad(x, (3,4))
        x = self.max_pool_2(x)
        x = self.dropout2(x)
        
        # ==== third block ====
        x = F.pad(x, (3,4))
        x = F.relu((self.conv3(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm3(self.conv3_3(x)))
#         x = F.relu(self.conv3_3(x))
        x = F.pad(x, (3,4))
        x = self.max_pool_3(x)
        x = self.dropout3(x)
        
        # ==== fourth block ====
        x = F.pad(x, (3,4))
        x = F.relu((self.conv4(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm4(self.conv4_4(x)))
#         x = F.relu(self.conv4_4(x))
        x = F.pad(x, (3,4))
        x = self.max_pool_4(x)
        x = self.dropout4(x)

                
        x = x.view(x.size(0), -1)
        
#         x = self.projection(x)

        x = self.fc(x)
                
        return x    
        

In [ ]:
class DFsimCLR(nn.Module):
    def __init__(self, df, out_dim):
        super(DFsimCLR, self).__init__()
        
        self.backbone = df
        self.backbone.weight_init()
        dim_mlp = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.BatchNorm1d(dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, out_dim)
        )
        
    def forward(self, inp):
        out = self.backbone(inp)
        return out

In [ ]:
def find_bursts(x):
    
    direction = x[0]
    bursts = []
    start = 0
    temp_burst = x[0]
    for i in range(1, len(x)):
        if x[i] == 0.0:
            break
        
        elif x[i] == direction:
            temp_burst += x[i]
            
        else:
            # if temp_burst <= -10 or temp_burst > 0:
            bursts.append((start, i, temp_burst))
            start = i
            temp_burst = x[i]
            direction *= -1
            
    return bursts

outgoing_burst_sizes = []

x_random = x_train[np.random.choice(range(len(x_train)), size=650, replace=True)]


for x in x_random:
    bursts = find_bursts(x)
    
    outgoing_burst_sizes += [x[2] for x in bursts if x[2] > 0]

max_outgoing_burst_size = max(outgoing_burst_sizes)

In [ ]:
count, bins = np.histogram(outgoing_burst_sizes, bins=max_outgoing_burst_size - 1)
PDF = count/np.sum(count)
OUTGOING_BURST_SIZE_CDF = np.zeros_like(bins)
OUTGOING_BURST_SIZE_CDF[1:] = np.cumsum(PDF)

In [ ]:
class Augmentor():
    def __init__(self):
        methods = {
            'merge downstream burst',
            'change downstream burst sizes',
            'merge downstream and upstream bursts',
            'add upstream bursts',
            'remove upstrean bursts',
            'divide bursts'
        }
        
        
        self.large_burst_threshold = 10
        
        # changing the content
        self.upsample_rate = 1.0
        self.downsample_rate = 0.5
        
        # merging bursts
        self.num_bursts_to_merge = 5
        self.merge_burst_rate = 0.1
        
        # add incoming bursts
        self.add_outgoing_burst_rate = 0.3
        self.outgoing_burst_sizes = list(range(max_outgoing_burst_size))
        
        # shift
        self.shift_param = 10
        
        
        
    def find_bursts(self, x):
        direction = x[0]
        bursts = []
        start = 0
        temp_burst = x[0]
        for i in range(1, len(x)):
            if x[i] == 0.0:
                break

            elif x[i] == direction:
                temp_burst += x[i]

            else:
                # if temp_burst <= -10 or temp_burst > 0:
                bursts.append((start, i, temp_burst))
                start = i
                temp_burst = x[i]
                direction *= -1

        return bursts
        
        
    # representing the change of contents of a website
    def increase_incoming_bursts(self, burst_sizes):
        out = []
        for i, size in enumerate(burst_sizes):
            if size <= -self.large_burst_threshold:
                up_sample_rate = random.random()*self.upsample_rate
                new_size = int(size * (1+up_sample_rate))
                out.append(new_size)
            else:
                out.append(size)
                
        return out
        
        
    def decrease_incoming_bursts(self, burst_sizes):
        out = []
        for i, size in enumerate(burst_sizes):
            if size <= -self.large_burst_threshold:
                up_sample_rate = random.random()*self.downsample_rate
                new_size = int(size * (1-up_sample_rate))
                out.append(new_size)
            else:
                out.append(size)
                
        return out
        
        
    def change_content(self, burst_sizes):  # receives burst_sizes, not trace
        # DON'T call find_bursts again - burst_sizes already passed in
        
        # Count non-zero packets to determine trace "length"
        total_packets = sum(abs(s) for s in burst_sizes)
        
        if total_packets < 1000:
            new_burst_sizes = self.increase_incoming_bursts(burst_sizes)
        elif total_packets > 4000:
            new_burst_sizes = self.decrease_incoming_bursts(burst_sizes)
        else:
            p = random.random()
            if p >= 0.5:
                new_burst_sizes = self.increase_incoming_bursts(burst_sizes)
            else:
                new_burst_sizes = self.decrease_incoming_bursts(burst_sizes)
        
        return new_burst_sizes
    
    
    def merge_incoming_bursts(self, burst_sizes):
        
        out = []
        
        # skipping first 20 cells
        i = 0
        num_cells = 0
        while i < len(burst_sizes) and num_cells < 20:
            num_cells += abs(burst_sizes[i])
            out.append(burst_sizes[i])
            i += 1
            
        
        while i < len(burst_sizes) - self.num_bursts_to_merge:
            prob = random.random()
            
            # ignore outgoing bursts
            if burst_sizes[i] > 0:
                out.append(burst_sizes[i])
                i+= 1
                continue
            
            if prob < self.merge_burst_rate:
                num_merges = random.randint(2, self.num_bursts_to_merge)
                merged_size = 0
                
                # merging the incoming bursts
                while i < len(burst_sizes) and num_merges > 0:
                    if burst_sizes[i] < 0:
                        merged_size += burst_sizes[i]
                        num_merges -= 1
                    i += 1     
                out.append(merged_size)
                    
            else:
                out.append(burst_sizes[i])
                i += 1
                
        return out
    
    
    def add_outgoing_burst(self, burst_sizes):
        
        out = []
        
        i = 0
        num_cells = 0
        while i < len(burst_sizes) and num_cells < 20:
            num_cells += abs(burst_sizes[i])
            out.append(burst_sizes[i])
            i += 1
            
        
        for size in burst_sizes[i:]:
            if size > -10 :
                out.append(size)
                continue
            
            prob = random.random()
            
            if prob < self.add_outgoing_burst_rate:
                
                index = len(outgoing_burst_sizes)
                while index >= len(outgoing_burst_sizes):
                    outgoing_burst_prob = random.random()
                    index = bisect.bisect_left(OUTGOING_BURST_SIZE_CDF, outgoing_burst_prob)
                    
                outgoing_burst_size = self.outgoing_burst_sizes[index]
                divide_place = random.randint(3, abs(size) - 3)
                
                out += [-divide_place, outgoing_burst_size, -(abs(size) - divide_place)]
                
            else:
                out.append(size)
                
        return out
                
        
    def create_trace_from_burst_sizes(self, burst_sizes):
        out = []
        
        for size in burst_sizes:
            val = 1 if size > 0 else -1
            
            out += [val]*(int(abs(size)))
            
        if len(out) < 10000:
            out += [0]*(10000 - len(out))
            
        return np.array(out)[:10000]
    
    def shift(self, x):
        pad = np.random.randint(0, 2, size = (self.shift_param, ))
        pad = 2*pad-1
        zpad = np.zeros_like(pad)
        
        shift_val = np.random.randint(-self.shift_param, self.shift_param+1, 1)[0]
        shifted = np.concatenate((x, zpad, pad), axis=-1)
        shifted = np.roll(shifted, shift_val, axis=-1)
        shifted = shifted[:10000]
        
        return shifted
        
    
    def augment(self, trace):
        bursts = self.find_bursts(trace)
        burst_sizes = [x[2] for x in bursts]
        
        if len(burst_sizes) == 0:  # guard against empty traces
            return self.shift(trace.copy())
        
        mapping = {
            0: self.change_content,
            1: self.merge_incoming_bursts,
            2: self.add_outgoing_burst
        }
        
        aug_method = mapping[random.randint(0, len(mapping)-1)]
        augmented_sizes = aug_method(burst_sizes)
        augmented_trace = self.create_trace_from_burst_sizes(augmented_sizes)
        
        return self.shift(augmented_trace)

In [ ]:
class TrainData(Dataset):
    def __init__(self, x_train, y_train, augmentor, n_views):
        self.x = x_train
        self.y = y_train
        self.augmentor = augmentor
        self.n_views = n_views
    
    def _aug(self, inp):
        flip_idx = np.random.randint(0, 4999, 250)
        x_w = inp.copy()
        temp = x_w[flip_idx]
        x_w[flip_idx] = x_w[flip_idx+1]
        x_w[flip_idx+1] = temp
        return x_w
    
    def __getitem__(self, index):
        return [self.augmentor.augment(self.x[index]) for i in range(self.n_views)], self.y[index]
    
    def __len__(self):
        return len(self.x)

In [ ]:
def accuracy(output, target, topk=(1,)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

In [ ]:
class NetCLR(object):
    def __init__(self, **args):
        self.model = args['model']
        self.optimizer = args['optimizer']
        self.scheduler = args['scheduler']
        self.fp16_precision = args['fp16_precision']
        self.num_epoches = args['num_epoches']
        self.batch_size = args['batch_size']
        self.device = args['device']
        self.temperature = args['temperature']
#         self.tester = args['tester']
        self.n_views = 2
        self.criterion = torch.nn.CrossEntropyLoss().to(self.device)
        self.log_every_n_step = 100
    
    def info_nce_loss(self, features):
        labels = torch.cat([torch.arange(self.batch_size) for i in range(self.n_views)], dim = 0)
        labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
        labels = labels.to(self.device)
        
        features = F.normalize(features, dim=1)
        
        similarity_matrix = torch.matmul(features, features.T)
        
        mask = torch.eye(labels.shape[0], dtype=torch.bool).to(self.device)
        labels = labels[~mask].view(labels.shape[0], -1)
        similarity_matrix = similarity_matrix[~mask].view(similarity_matrix.shape[0], -1)
        
        positives = similarity_matrix[labels.bool()].view(labels.shape[0], -1)
        
        
        negatives = similarity_matrix[~labels.bool()].view(similarity_matrix.shape[0], -1)
        
        
        logits = torch.cat([positives, negatives], dim=1)
        labels = torch.zeros(logits.shape[0], dtype=torch.long).to(self.device)
        
        logits = logits / self.temperature
        return logits, labels
        
    def train(self, train_loader):
        best_acc = 0
        scaler = GradScaler(enabled=self.fp16_precision)

        n_iter = 0
        print ("Start SimCLR training for %d number of epoches"%self.num_epoches)
        
        first_loss = True
        for epoch_counter in range(self.num_epoches+1):
            
#             print ("Epoch: ", epoch_counter)
            with tqdm.tqdm(train_loader, unit='batch') as tepoch:
                for data, _ in tepoch:
                    tepoch.set_description(f"Epoch {epoch_counter}")
                    
                    model.train()
                    data = torch.cat(data, dim = 0)
                    data = data.view(data.size(0), 1, data.size(1))
                    data = data.float().to(self.device)

                    with autocast(enabled=self.fp16_precision):
                        features = self.model(data)
                        logits, labels = self.info_nce_loss(features)
                        loss = self.criterion(logits, labels)

                    self.optimizer.zero_grad()
                    
                    scaler.scale(loss).backward()
                    scaler.step(self.optimizer)
                    scaler.update()
                    
                    if n_iter%self.log_every_n_step == 0:
                        top1, top5 = accuracy(logits, labels, topk=(1, 5))
                        tepoch.set_postfix(loss=loss.item(), accuracy = top1.item())
                    n_iter += 1

            if epoch_counter >= 10:
                self.scheduler.step()
                
            if epoch_counter % 10 == 0 and epoch_counter != 0:
                os.makedirs('/kaggle/working/models/NetCLR', exist_ok=True)
                torch.save(self.model.state_dict(), f'/kaggle/working/models/NetCLR/NetCLR.pth.tar')
                print(f"\n💾 Model saved: /kaggle/working/models/NetCLR/NetCLR.pth.tar")

In [ ]:
temperature = 0.5 # this value is suggested by the original SimCLR paper
augmentor = Augmentor()

train_dataset = TrainData(x_train, y_train, augmentor, 2)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

df = DFNet(out_dim=512)
model = DFsimCLR(df, out_dim=128).to(device)


optimizer = torch.optim.Adam(model.parameters(), lr=0.0003) #, weight_decay = 1e-6)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader), eta_min=0, last_epoch=-1)

# Tạo folder để save model trên Kaggle
os.makedirs('/kaggle/working/models/NetCLR', exist_ok=True)
print("✓ Model directory created: /kaggle/working/models/NetCLR\n")

netclr = NetCLR(model = model,
               optimizer = optimizer,
               scheduler = scheduler,
               fp16_precision = fp16_precision,
               device = device,
               temperature = temperature,
               n_views = n_views,
               num_epoches = 70,
               batch_size = batch_size)
netclr.train(train_loader)

print("\n✅ Pre-training completed!")
print(f"✓ Models saved in: ./models/NetCLR/")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Split data: 50% train, 50% test
x_train_total, x_test, y_train_total, y_test = train_test_split(
    x_train, y_train, test_size=0.5, random_state=42, stratify=y_train
)


def sample_traces(x_data, y_data, n_samples_per_class):
    """Sample N traces per class for few-shot learning."""
    x_sampled, y_sampled = [], []
    
    for label in np.unique(y_data):
        indices = np.where(y_data == label)[0]
        sampled_indices = np.random.choice(
            indices, size=min(n_samples_per_class, len(indices)), replace=False
        )
        x_sampled.append(x_data[sampled_indices])
        y_sampled.append(y_data[sampled_indices])
    
    return np.concatenate(x_sampled), np.concatenate(y_sampled)


class Data(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    
    def __len__(self):
        return len(self.x)


def load_pretrained_model(checkpoint_path):
    """Load pre-trained NetCLR model, stripping backbone prefix and fc layers."""
    model_ft = DFNet(out_dim=num_classes).to(device)
    checkpoint = torch.load(checkpoint_path)
    
    new_checkpoint = {
        k[len("backbone."):]: v
        for k, v in checkpoint.items()
        if k.startswith('backbone.') and not k.startswith('backbone.fc')
    }
    
    model_ft.load_state_dict(new_checkpoint, strict=False)
    return model_ft


def train_finetune(model, device, train_loader, optimizer):
    """One epoch of fine-tuning."""
    model.train()
    total_loss = 0
    for data, target in train_loader:
        data = data.view(data.size(0), 1, data.size(1)).float().to(device)
        target = target.to(device)
        
        optimizer.zero_grad()
        loss = F.cross_entropy(model(data), target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(train_loader)


def test_finetune(model, device, loader):
    """Evaluate model accuracy on a data loader."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data = data.view(data.size(0), 1, data.size(1)).float().to(device)
            target = target.to(device)
            pred = model(data).argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).float().sum().item()
    
    return correct / len(loader.dataset)


def test_finetune_with_report(model, device, loader):
    """Evaluate model and return predictions for detailed metrics."""
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for data, target in loader:
            data = data.view(data.size(0), 1, data.size(1)).float().to(device)
            target = target.to(device)
            all_preds.extend(model(data).argmax(dim=1).cpu().numpy())
            all_targets.extend(target.cpu().numpy())
    
    accuracy = np.mean(np.array(all_preds) == np.array(all_targets))
    return accuracy, all_preds, all_targets


# Fine-tuning config
N_SAMPLES = 50
NUM_RUNS = 1
NUM_EPOCHS = 100
PRETRAINED_MODEL_PATH = '/kaggle/working/models/NetCLR/NetCLR.pth.tar'

test_dataset = Data(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

accuracies = []

for run in range(NUM_RUNS):
    x_train_sampled, y_train_sampled = sample_traces(x_train_total, y_train_total, N_SAMPLES)
    
    train_loader = DataLoader(
        Data(x_train_sampled, y_train_sampled),
        batch_size=batch_size, shuffle=True, drop_last=True
    )
    
    model = load_pretrained_model(PRETRAINED_MODEL_PATH)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    best_acc = 0
    
    for epoch in range(NUM_EPOCHS + 1):
        if epoch > 0:
            train_finetune(model, device, train_loader, optimizer)
        
        if epoch % 10 == 0:
            acc = test_finetune(model, device, test_loader)
            best_acc = max(best_acc, acc)
            print(f"Epoch {epoch:3d} | Accuracy: {acc*100:.2f}% | Best: {best_acc*100:.2f}%")

    final_acc = test_finetune(model, device, test_loader)
    best_acc = max(best_acc, final_acc)
    accuracies.append(best_acc)

# Final results
final_acc, preds, targets = test_finetune_with_report(model, device, test_loader)
print(f"Final Accuracy: {final_acc*100:.2f}%")
print(classification_report(targets, preds, digits=4))

mean_acc = np.mean(accuracies) * 100
std_acc = np.std(accuracies) * 100
print(f"Mean Accuracy: {mean_acc:.2f}% ± {std_acc:.2f}%")